<a href="https://colab.research.google.com/github/Garden-Tree/alpp/blob/main/Backpropagation%EF%BC%88%E8%AA%A4%E5%B7%AE%E9%80%86%E4%BC%9D%E6%92%AD%E6%B3%95%EF%BC%89.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# [W07]  04 - Backpropagation（誤差逆伝播法）

これまでの講義で、PyTorchの基礎である「テンソル（Tensor）」の操作と、自動微分機能である「Autograd」について学びました。
今回は、ニューラルネットワークの学習プロセスの中核をなす **バックプロパゲーション（誤差逆伝播法）** のアルゴリズムを解説し、PyTorchでの具体的な実装方法をハンズオン形式で学びます。

## 本講義のアジェンダ
1. 計算グラフと連鎖律（Chain Rule）
2. 順伝播（Forward pass）と逆伝播（Backward pass）
3. 単回帰（Linear Regression）を用いた具体例とコード実装

## 1. 計算グラフと連鎖律（Chain Rule）
ニューラルネットワークの学習は、損失（Loss）を最小化するようにパラメータ（重み）を調整する最適化問題です。そのために「パラメータを微小に動かしたとき、Lossがどれくらい変化するか」を示す **勾配（Gradient）** を求める必要があります。

ディープラーニングでは、複数の関数が合成された複雑な数式を扱いますが、計算全体をノード（変数）とエッジ（演算）からなる「計算グラフ」として表現することで、微分の公式である **連鎖律（Chain Rule）** を適用できます。

連鎖律とは、合成関数の微分を各部分の微分の積で表す法則です。
$$\frac{\partial z}{\partial x} = \frac{\partial z}{\partial y} \cdot \frac{\partial y}{\partial x}$$

## 2. 順伝播と逆伝播
計算グラフにおいて、学習は2つのパス（経路）で行われます。

|プロセス|方向|行うこと|
|-|-|-|
|Forward pass (順伝播)|入力 $\rightarrow$ 出力|入力データと現在の重みから予測値を計算し、<br>最終的に損失（Loss）を算出する。|
|Backward pass (逆伝播)|出力 $\rightarrow$ 入力|Lossから出発し、連鎖律を用いて各ノードの局所的な勾配（Local gradients）を掛け合わせながら、<br>各重みに対する勾配を算出する。|

## 3. 単回帰を用いた具体例とコード実装
ここでは、非常にシンプルな単回帰モデル $y = w \cdot x$ を例に、パラメータ $w$ を最適化する過程をPyTorchで実装します。

* 入力値: $x = 1.0$
* 正解値: $y = 2.0$
* 初期重み: $w = 1.0$

損失関数には平均二乗誤差（の1データ分）を用います。
$$Loss = (\hat{y} - y)^2 = (w \cdot x - y)^2$$

## 【演習】PyTorchの気持ちになって、手計算で逆伝播してみよう！

PyTorchの `backward()` を呼ぶ前に、私たちが自分で計算グラフを逆走して、勾配 $\frac{\partial Loss}{\partial w}$ を求めてみましょう。

**現在の状態（順伝播完了時点）:**
* $x = 1.0, \quad y = 2.0, \quad w = 1.0$
* $\hat{y} = w \cdot x = 1.0$
* $Loss = (\hat{y} - y)^2 = (1.0 - 2.0)^2 = 1.0$

---

### Step 1: 最初の逆伝播（Lossの微分）
まずは、一番右端の $Loss$ を $\hat{y}$ で微分します。
$$\frac{\partial Loss}{\partial \hat{y}} = 2 \cdot (\hat{y} - y)$$
ここに数値を代入すると…
$$\frac{\partial Loss}{\partial \hat{y}} = 2 \cdot (1.0 - 2.0) = \mathbf{-2.0}$$

### Step 2: 次の逆伝播（$\hat{y}$ の微分）
次に、$\hat{y} = w \cdot x$ を、求めたい $w$ で微分します。
$$\frac{\partial \hat{y}}{\partial w} = x$$
数値を代入すると…
$$\frac{\partial \hat{y}}{\partial w} = \mathbf{1.0}$$

### Step 3: 連鎖律（Chain Rule）で合体！
最後に、Step1とStep2の結果を掛け合わせます。これが逆伝播（Backpropagation）の正体です！
$$\frac{\partial Loss}{\partial w} = \frac{\partial Loss}{\partial \hat{y}} \cdot \frac{\partial \hat{y}}{\partial w} = (-2.0) \cdot (1.0) = \mathbf{-2.0}$$

これで、勾配は **-2.0** になるはずだと分かりました。
では、PyTorchの `backward()` が本当にこの通りに計算してくれるか、コードで答え合わせをしてみましょう！

In [1]:
import torch

# 1. データの準備
x = torch.tensor(1.0)
y = torch.tensor(2.0)

# 2. 重みの定義（最適化対象のパラメータであるため requires_grad=True を指定）
w = torch.tensor(1.0, requires_grad=True)

# 3. 順伝播（Forward pass）: 予測値と損失の計算
y_predicted = w * x
loss = (y_predicted - y)**2

print(f"Forward pass後の Loss: {loss.item()}")

# 4. 逆伝播（Backward pass）: 勾配 dLoss/dw の計算
loss.backward()

print(f"Backward pass後の 勾配 (dLoss/dw): {w.grad.item()}")

Forward pass後の Loss: 1.0
Backward pass後の 勾配 (dLoss/dw): -2.0
